## Basic Breakpoint Detection for historical time series

### Breakpoint Methods
* Beast
* Simple Breakpoint Detection

In [ ]:
# imports
from pathlib import Path

import xarray as xr

from water_timeseries.breakpoint import BeastBreakpoint, SimpleBreakpoint, NRTBreakpoint
from water_timeseries.dataset import DWDataset, JRCDataset

## Load data

In [ ]:
def dw_test_dataset():
    """Load the real Dynamic World test dataset.

    Returns the actual DW dataset from tests/data/lakes_dw_test.nc
    """
    test_data_dir = Path("../tests/data")
    dw_file = test_data_dir / "lakes_dw_test.nc"
    return xr.open_dataset(dw_file)


def jrc_test_dataset():
    """Load the real JRC water test dataset.

    Returns the actual JRC dataset from tests/data/lakes_jrc_test.nc
    """
    test_data_dir = Path("../tests/data")
    jrc_file = test_data_dir / "lakes_jrc_test.nc"
    return xr.open_dataset(jrc_file)

Create water Datasets:
* JRC 
* Dynamic World

In [ ]:
jrc_ds = JRCDataset(jrc_test_dataset())
dw_ds = DWDataset(dw_test_dataset())

Show valid ids
* both have the same lake datasets
* JRC has annual data 1986-2020
* Dynamic World has monthly data (June-September) 2015-2024

In [ ]:
print(jrc_ds.object_ids_)
print(jrc_ds.dates_)

print(dw_ds.object_ids_)
print(dw_ds.dates_)

## Data Exploration
Lake "b7uefy0bvcrc" drained in early 2018
* lets plot the time series for both JRC and DW datasets
* it shows a clear drop in 2018 in both datasets

In [ ]:
jrc_ds.plot_timeseries_interactive(id_geohash="b7uefy0bvcrc")

In [ ]:
dw_ds.plot_timeseries_interactive(id_geohash="b7uefy0bvcrc")

## Breakpoint Detection

For the historical analysis, if and when did we observe lake area loss/drainage, we have 2 methods
* BeastBreakpoint based on the RBeast Breakpoint method
* SimpleBreakpoint which sind uses a rolling mean with a threshold

#### RBeast Break
This method/function calculates a potential breakpoint. It contains several attributes
* date_break: breakpoint in datetime format (such as the "date" dimension in the input dataset) + dates before and after break for later analysis
* break_number: sorted ascending id by intensity (determined by RBeast) with 1 the most significant break, 2 less, ..n
* pre_break_...: water area stats before break
* post_break_...: water area stats after break
* water_change_..: water change stats after vs before in ha and %

In [ ]:
# setting up breakpoint method
bp_beast = BeastBreakpoint()

Dynamic World Dataset

In [ ]:
# this checks for breakpoints in the timeseries of the given geohash and returns a Dataframe of breakpoints and specific statistics
breaks_beast_dw = bp_beast.calculate_break(dw_ds, object_id="b7uefy0bvcrc")
breaks_beast_dw

JRC Dataset
* Beast finds the breakpoint for 2018-01-01, since JRC dataset is indexed at this date
* output structure is the same regardless of the Datsetset

In [ ]:
# this checks for breakpoints in the timeseries of the given geohash and returns a Dataframe of breakpoints and specific statistics
breaks_beast_jrc = bp_beast.calculate_break(jrc_ds, object_id="b7uefy0bvcrc")
breaks_beast_jrc

#### Simple Breakpoint
* it allows to choose a rolling aggregation method window size and threshold
* https://permafrostdiscoverygateway.github.io/water-timeseries-v2/api/breakpoint/#water_timeseries.breakpoint.SimpleBreakpoint--attributes

In [ ]:
# setting up breakpoint method
bp_simple = SimpleBreakpoint(kwargs_break=dict(method="median", window=3, threshold=-0.25))

In [ ]:
breaks_simple_dw = bp_simple.calculate_break(dw_ds, object_id="b7uefy0bvcrc")
breaks_simple_dw

In [ ]:
breaks_simple_jrc = bp_simple.calculate_break(jrc_ds, object_id="b7uefy0bvcrc")
breaks_simple_jrc

### Near Real-time analysis
* NRT analysis uses ARIMA to determine if a specified date (typically the latest date) is an outlier compared to the previous time-series

In [ ]:
bp_nrt = NRTBreakpoint()

Here we can check the ARIMA prediction for the individual lake
* it contains stats on the predicted and observed water area
* it further contains prediction confidences (p10/p90), and
* historical water area stats

The water_residual of **-0.8074** indicates an 80% lower water area than predicted
* with a 90% confidence it is expected that water area should be between 0.9873 and 0.9936


In [ ]:
bp_nrtbreaks_simple_dw = bp_nrt.calculate_break(
    dataset=dw_ds, analysis_date="2018-06-01", data_aggregation_period="all", object_id="b7uefy0bvcrc"
)
bp_nrtbreaks_simple_dw

We can also check another date - e.g. 2019-07-01
* water_residual is close to 0 -> no drainage

In [ ]:
bp_nrtbreaks_simple_dw = bp_nrt.calculate_break(
    dataset=dw_ds, analysis_date="2019-07-01", data_aggregation_period="all", object_id="b7uefy0bvcrc"
)
bp_nrtbreaks_simple_dw

It also works for the JRC dataset
* water_residual is **-0.5146**

In [ ]:
bp_nrtbreaks_simple_jrc = bp_nrt.calculate_break(
    dataset=jrc_ds, analysis_date="2018-01-01", data_aggregation_period="all", object_id="b7uefy0bvcrc"
)
bp_nrtbreaks_simple_jrc

#### Loop
We can also run over the entire dataset in parallel

In [ ]:
bp_nrtbreaks_simple_dw = bp_nrt.calculate_break(
    dataset=dw_ds, analysis_date="2019-07-01", data_aggregation_period="all", object_id=None
)
bp_nrtbreaks_simple_dw